In [ ]:
import networkit as nk
import pandas as pd
import time
from pathlib import Path

# -- Change these two values before running ------------------------------------
SNAPSHOT = "2026-06-30"
QUERY    = "sustainability"
# ------------------------------------------------------------------------------

# Derive input and output paths from SNAPSHOT / QUERY
graph_file  = Path("input")  / f"snapshot_{SNAPSHOT}" / QUERY / "edges.txt"
output_dir  = Path("output") / f"snapshot_{SNAPSHOT}" / QUERY
output_file = output_dir / "louvain_clusters.txt"

# Create output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Snapshot: {SNAPSHOT}")
print(f"Query   : {QUERY}")
print(f"Input   : {graph_file}")
print(f"Output  : {output_file}")

# --- 1. Setup ---
print("\nStarting clustering process...")

# Set the number of threads for NetworKit to use.
# Match this with the number of vCPUs on your machine.
nk.setNumberOfThreads(60)
print(f"NetworKit is configured to use {nk.getMaxNumberOfThreads()} threads.")

Input  : input/sustainability/version3/edges_dedup.txt
Output : output/sustainability/version3/louvain_clusters.txt

Starting clustering process...
NetworKit is configured to use 60 threads.


In [ ]:
# --- 2. Load the Full Graph ---
print(f"\nStep 1/6: Loading full graph from '{graph_file}'...")
start_time = time.time()

# edges.txt is produced by step 03 and downloaded by download_input.ipynb.
G = nk.graphio.readGraph(str(graph_file), fileformat=nk.Format.EdgeListTabZero,
                         directed=False)

end_time = time.time()
print(f"-> Full graph loaded in {end_time - start_time:.2f} seconds.")
print(f"-> Graph has {G.numberOfNodes():,} nodes and {G.numberOfEdges():,} edges.")


Step 1/6: Loading full graph from 'input/sustainability/version3/edges_dedup.txt'...
-> Full graph loaded in 0.45 seconds.
-> Graph has 310,600 nodes and 1,677,645 edges.


In [3]:
# --- 3. Find and Extract the Largest Connected Component ---
print("\nStep 2/6: Finding the largest connected component...")
start_time = time.time()

# Calculate connected components
cc = nk.components.ConnectedComponents(G)
cc.run()
component_partition = cc.getPartition()

# Identify the largest component by finding the partition subset with the most members
largest_component_id = max(component_partition.subsetSizeMap(), key=component_partition.subsetSizeMap().get)
largest_component_nodes = component_partition.getMembers(largest_component_id)

end_time = time.time()
print(f"-> Component analysis finished in {end_time - start_time:.2f} seconds.")




Step 2/6: Finding the largest connected component...
-> Component analysis finished in 0.11 seconds.


In [4]:
print("\nStep 3/6: Creating subgraph of the largest component...")
start_time = time.time()
# Create a subgraph containing only the nodes of the largest component.
# This function crucially PRESERVES the original node IDs.
G_lcc = nk.graphtools.subgraphFromNodes(G, nodes=largest_component_nodes, compact=False)

end_time = time.time()
print(f"-> Subgraph created for the largest component.")
print(f"-> The largest component contains {G_lcc.numberOfNodes():,} nodes and {G_lcc.numberOfEdges():,} edges.")
print(f"-> Subgraph created in {end_time - start_time:.2f} seconds.")



Step 3/6: Creating subgraph of the largest component...
-> Subgraph created for the largest component.
-> The largest component contains 300,097 nodes and 1,671,010 edges.
-> Subgraph created in 0.17 seconds.


---

## 🔴 Micro-level

In [ ]:
# --- 4. Run Clustering on the Largest Component ---
print("\nStep 4/6: Running Parallel Louvain Method (PLM) on the largest component...")
start_time = time.time()

# Instantiate the PLM algorithm on the subgraph

# For the large graphs >10 million nodes:
#micro_plm = nk.community.PLM(G_lcc, refine=True, gamma=5000, par='balanced', maxIter=50)
#micro_plm = nk.community.LouvainMapEquation(G_lcc, hierarchical=True, maxIterations=50)

# For networks with <1 million and > 100k nodes, you can use the default PLM settings:
micro_plm = nk.community.PLM(G_lcc, refine=True, gamma=70, par='balanced', maxIter=50)

micro_plm.run()
micro_partition = micro_plm.getPartition()

# IMPORTANT: compact subset IDs before building higher-level community graphs.
# This prevents sparse micro IDs from breaking micro->meso mapping later.
micro_partition.compact()

end_time = time.time()
modularity = nk.community.Modularity().getQuality(micro_partition, G_lcc)

print(f"-> Clustering completed in {end_time - start_time:.2f} seconds.")
print(f"-> Found {micro_partition.numberOfSubsets():,} communities with a modularity of {modularity:.4f}.")
print("-> Micro partition IDs were compacted to a dense range [0..K-1].")


Step 4/6: Running Parallel Louvain Method (PLM) on the largest component...
-> Clustering completed in 0.38 seconds.
-> Found 461 communities with a modularity of 0.5360.
-> Micro partition IDs were compacted to a dense range [0..K-1].


In [6]:
# --- 6. Analyze and Display Top Cluster Sizes ---
print("\nStep 6/6: Analyzing sizes of the largest communities...")
start_time = time.time()

# Get a map of {community_id: size}
community_sizes_map = micro_partition.subsetSizeMap()

# Get a list of all community sizes
all_sizes = list(community_sizes_map.values())

# Sort the sizes in descending order
all_sizes.sort()

# --- Display Top 20 Largest ---
# Get the last 20 elements from the sorted list
top_20_sizes = all_sizes[-20:]
# Reverse the new list to display from largest to smallest
top_20_sizes.reverse()

print("\n--- Top 20 Largest Communities ---")
for i, size in enumerate(top_20_sizes, 1):
    print(f"Rank {i:<2}: {size:,} members")




Step 6/6: Analyzing sizes of the largest communities...

--- Top 20 Largest Communities ---
Rank 1 : 6,707 members
Rank 2 : 6,049 members
Rank 3 : 4,803 members
Rank 4 : 4,406 members
Rank 5 : 4,223 members
Rank 6 : 4,091 members
Rank 7 : 3,916 members
Rank 8 : 3,915 members
Rank 9 : 3,519 members
Rank 10: 3,507 members
Rank 11: 3,444 members
Rank 12: 3,271 members
Rank 13: 3,236 members
Rank 14: 3,105 members
Rank 15: 3,033 members
Rank 16: 2,911 members
Rank 17: 2,887 members
Rank 18: 2,781 members
Rank 19: 2,747 members
Rank 20: 2,685 members


In [7]:
# --- Display Bottom 20 Smallest ---
# Get the first 20 elements from the sorted list
bottom_20_sizes = all_sizes[:20]

print("\n--- Bottom 20 Smallest Communities ---")
for i, size in enumerate(bottom_20_sizes, 1):
    print(f"Rank {i:<2}: {size:,} members")


end_time = time.time()
print(f"-> Successfully summarized cluster sizes in {end_time - start_time:.2f} seconds.")

print("\nProcess complete. ✨")


--- Bottom 20 Smallest Communities ---
Rank 1 : 2 members
Rank 2 : 2 members
Rank 3 : 2 members
Rank 4 : 3 members
Rank 5 : 3 members
Rank 6 : 3 members
Rank 7 : 3 members
Rank 8 : 3 members
Rank 9 : 3 members
Rank 10: 3 members
Rank 11: 3 members
Rank 12: 3 members
Rank 13: 3 members
Rank 14: 3 members
Rank 15: 3 members
Rank 16: 3 members
Rank 17: 3 members
Rank 18: 3 members
Rank 19: 3 members
Rank 20: 3 members
-> Successfully summarized cluster sizes in 0.02 seconds.

Process complete. ✨


In [8]:
# --- Display Number of clusters more than x ---
thr_cls = [num for num in all_sizes if num >= 50]
thr_cnt = sum(thr_cls)
print(f"\nNumber of clusters with more than 50 members: {len(thr_cls):,} | {len(thr_cls) / len(all_sizes) * 100:.2f}% of all clusters")
print(f"They cover {thr_cnt / sum(all_sizes) * 100:.2f}% of all members in the largest component.")


Number of clusters with more than 50 members: 347 | 75.27% of all clusters
They cover 99.73% of all members in the largest component.


In [9]:
# --- 5. Write Results to File ---
print(f"\nStep 5/6: Writing community assignments to '{output_file}'...")
start_time = time.time()

count = 0
with open(output_file, 'w') as f:
    # We iterate through the nodes of the largest component subgraph.
    # The partition object `micro_partition` uses the original node IDs.
    for node_id in G_lcc.iterNodes():
        community_id = micro_partition[node_id]
        f.write(f"{node_id}\t{community_id}\n")
        count += 1

end_time = time.time()
print(f"-> Successfully wrote {count:,} assignments in {end_time - start_time:.2f} seconds.")




Step 5/6: Writing community assignments to 'output/sustainability/version3/louvain_clusters.txt'...
-> Successfully wrote 300,097 assignments in 0.08 seconds.


---
## 🔴 Meso-level

In [ ]:
# --- 4. Meso-Level Clustering ---
print("\n---Performing MESO-level clustering...")
start_time = time.time()

# Create a new graph where each node is a micro-cluster.
# Edge weights between clusters are the sum of weights of the original edges connecting them.
G_meso = nk.community.communityGraph(G_lcc, micro_partition)
#G_meso = nk.community.GraphClusteringTools.communicationGraph(G_lcc, micro_partition)

# Run PLM on the new, smaller community graph
meso_plm = nk.community.PLM(G_meso, refine=False, gamma=25) #100 for the large graphs >10 million nodes, 10 for the smaller graphs
meso_plm.run()
meso_partition = meso_plm.getPartition()

# Keep meso labels dense for robust meso->macro mapping.
meso_partition.compact()

end_time = time.time()
print(f"-> MESO-level clustering found {meso_partition.numberOfSubsets():,} communities in {end_time - start_time:.2f}s.")
print("-> Meso partition IDs were compacted to a dense range [0..K-1].")




---Performing MESO-level clustering...
-> MESO-level clustering found 42 communities in 0.01s.
-> Meso partition IDs were compacted to a dense range [0..K-1].


---
## 🔴 Macro - level

In [ ]:
# --- 5. Macro-Level Clustering ---
print("\n---Performing MACRO-level clustering...")
start_time = time.time()

# Create a final graph where each node is a meso-cluster
G_macro = nk.community.communityGraph(G_meso, meso_partition)

# Run PLM on this final, tiny graph
macro_plm = nk.community.PLM(G_macro, refine=False, gamma=5) #10 for the large graphs >10 million nodes, 1 for the smaller graphs
macro_plm.run()
macro_partition = macro_plm.getPartition()

# Keep macro labels dense for stable downstream use.
macro_partition.compact()

end_time = time.time()
print(f"-> MACRO-level clustering found {macro_partition.numberOfSubsets():,} communities in {end_time - start_time:.2f}s.")
print("-> Macro partition IDs were compacted to a dense range [0..K-1].")


---Performing MACRO-level clustering...
-> MACRO-level clustering found 12 communities in 0.00s.
-> Macro partition IDs were compacted to a dense range [0..K-1].


---
## Write results

In [12]:
# --- 6. Map and Write Results (robust hierarchy checks) ---
# This version is fail-fast against ID-space mismatches across levels.
# It relies on compacted partition IDs and validates that communityGraph node IDs
# align with those compacted IDs before any merge/write step.

import numpy as np

print("\n--- Mapping hierarchical assignments (robust) ---")
start_time = time.time()

# -- Step A: node IDs present in the LCC (preserves original node IDs).
node_ids = np.fromiter(G_lcc.iterNodes(), dtype=np.int64, count=G_lcc.numberOfNodes())

# -- Step B: partition vectors (index = graph node id, value = assigned cluster id).
#    uint64 is required because nk.none = 2^64 - 1.
micro_vec = np.array(micro_partition.getVector(), dtype=np.uint64)
meso_vec  = np.array(meso_partition.getVector(), dtype=np.uint64)
macro_vec = np.array(macro_partition.getVector(), dtype=np.uint64)
none_sentinel = np.uint64(nk.none)

# Ensure every article node got a micro assignment.
micro_assignments = micro_vec[node_ids]
if np.any(micro_assignments == none_sentinel):
    bad = int(np.sum(micro_assignments == none_sentinel))
    raise RuntimeError(f"Found {bad:,} nodes without micro assignment (nk.none).")

# -- Step C: graph-node domains at higher levels.
meso_node_ids  = np.fromiter(G_meso.iterNodes(), dtype=np.int64, count=G_meso.numberOfNodes())
macro_node_ids = np.fromiter(G_macro.iterNodes(), dtype=np.int64, count=G_macro.numberOfNodes())

# Validate expected node-ID domains before joining levels.
expected_micro_ids = np.arange(micro_partition.numberOfSubsets(), dtype=np.int64)
if not np.array_equal(np.sort(meso_node_ids), expected_micro_ids):
    raise RuntimeError(
        "micro->meso ID mismatch: G_meso nodes do not match compact micro IDs. "
        "Aborting to avoid silent misassignment."
    )

expected_meso_ids = np.arange(meso_partition.numberOfSubsets(), dtype=np.int64)
if not np.array_equal(np.sort(macro_node_ids), expected_meso_ids):
    raise RuntimeError(
        "meso->macro ID mismatch: G_macro nodes do not match compact meso IDs. "
        "Aborting to avoid silent misassignment."
    )

# -- Step D: build mapping tables (safe because domains were validated).
meso_map = pd.DataFrame({
    "micro_cluster": meso_node_ids.astype(np.int64),
    "meso_cluster": meso_vec[meso_node_ids].astype(np.int64),
})

macro_map = pd.DataFrame({
    "meso_cluster": macro_node_ids.astype(np.int64),
    "macro_cluster": macro_vec[macro_node_ids].astype(np.int64),
})

# -- Step E: map node -> micro -> meso -> macro.
df = pd.DataFrame({
    "node_id": node_ids,
    "micro_cluster": micro_assignments.astype(np.int64),
})

df = (
    df
    .merge(meso_map, on="micro_cluster", how="left")
    .merge(macro_map, on="meso_cluster", how="left")
)[["node_id", "micro_cluster", "meso_cluster", "macro_cluster"]]

# Final guard: any null here indicates broken cross-level linkage.
null_meso = int(df["meso_cluster"].isna().sum())
null_macro = int(df["macro_cluster"].isna().sum())
if null_meso > 0 or null_macro > 0:
    raise RuntimeError(
        f"Hierarchy mapping produced nulls: meso={null_meso:,}, macro={null_macro:,}."
    )

end_time = time.time()
print(f"-> Mapping complete in {end_time - start_time:.2f} seconds.")
print(f"-> Null checks passed: meso={null_meso:,}, macro={null_macro:,}")

# -- Step F: write to file.
print(f"\nWriting results to '{output_file}' ...")
start_time = time.time()
df.to_csv(output_file, sep="\t", header=True, index=False)
end_time = time.time()
print(f"-> Successfully wrote {len(df):,} assignments in {end_time - start_time:.2f} seconds.")
print("\nProcess complete. ✨")


--- Mapping hierarchical assignments (robust) ---
-> Mapping complete in 0.05 seconds.
-> Null checks passed: meso=0, macro=0

Writing results to 'output/sustainability/version3/louvain_clusters.txt' ...
-> Successfully wrote 300,097 assignments in 0.15 seconds.

Process complete. ✨


In [13]:
# --- 7. Post-write hierarchy QC summary ---
# Run this after the mapping/write cell to validate hierarchy integrity quickly.

import pandas as pd

if "df" not in globals():
    raise RuntimeError("DataFrame 'df' is not available. Run the mapping cell first.")

qc = df[["node_id", "micro_cluster", "meso_cluster", "macro_cluster"]].copy()

docs_total = len(qc)
null_meso = int(qc["meso_cluster"].isna().sum())
null_macro = int(qc["macro_cluster"].isna().sum())

micro_to_meso = qc.groupby("micro_cluster")["meso_cluster"].nunique(dropna=True)
micro_to_macro = qc.groupby("micro_cluster")["macro_cluster"].nunique(dropna=True)
meso_to_macro = qc.groupby("meso_cluster")["macro_cluster"].nunique(dropna=True)

micro_without_meso = int((micro_to_meso == 0).sum())
micro_without_macro = int((micro_to_macro == 0).sum())
micro_with_multi_meso = int((micro_to_meso > 1).sum())
micro_with_multi_macro = int((micro_to_macro > 1).sum())
meso_without_macro = int((meso_to_macro == 0).sum())
meso_with_multi_macro = int((meso_to_macro > 1).sum())

summary = pd.DataFrame([
    {
        "docs_total": docs_total,
        "null_meso_docs": null_meso,
        "null_macro_docs": null_macro,
        "micro_clusters": int(qc["micro_cluster"].nunique()),
        "meso_clusters": int(qc["meso_cluster"].nunique()),
        "macro_clusters": int(qc["macro_cluster"].nunique()),
        "micro_without_meso": micro_without_meso,
        "micro_without_macro": micro_without_macro,
        "micro_with_multiple_meso": micro_with_multi_meso,
        "micro_with_multiple_macro": micro_with_multi_macro,
        "meso_without_macro": meso_without_macro,
        "meso_with_multiple_macro": meso_with_multi_macro,
    }
])

print("\n--- Hierarchy QC Summary ---")
print(summary.to_string(index=False))

if micro_with_multi_meso or micro_with_multi_macro or meso_with_multi_macro or null_meso or null_macro:
    print("\n[warn] Potential hierarchy issues detected. Showing top problematic IDs:")

    bad_micro = (
        qc.groupby("micro_cluster", as_index=False)
        .agg(
            docs=("node_id", "count"),
            meso_distinct=("meso_cluster", lambda s: s.nunique(dropna=True)),
            macro_distinct=("macro_cluster", lambda s: s.nunique(dropna=True)),
            null_meso_docs=("meso_cluster", lambda s: int(s.isna().sum())),
            null_macro_docs=("macro_cluster", lambda s: int(s.isna().sum())),
        )
    )
    bad_micro = bad_micro[
        (bad_micro["meso_distinct"] > 1)
        | (bad_micro["macro_distinct"] > 1)
        | (bad_micro["null_meso_docs"] > 0)
        | (bad_micro["null_macro_docs"] > 0)
    ].sort_values(["docs", "micro_cluster"], ascending=[False, True])
    if not bad_micro.empty:
        print("\n[bad micro] top 20")
        print(bad_micro.head(20).to_string(index=False))

    bad_meso = (
        qc.groupby("meso_cluster", as_index=False)
        .agg(
            docs=("node_id", "count"),
            macro_distinct=("macro_cluster", lambda s: s.nunique(dropna=True)),
            null_macro_docs=("macro_cluster", lambda s: int(s.isna().sum())),
        )
    )
    bad_meso = bad_meso[
        (bad_meso["macro_distinct"] > 1)
        | (bad_meso["null_macro_docs"] > 0)
    ].sort_values(["docs", "meso_cluster"], ascending=[False, True])
    if not bad_meso.empty:
        print("\n[bad meso] top 20")
        print(bad_meso.head(20).to_string(index=False))
else:
    print("\n[ok] Hierarchy mapping checks passed (no nulls and one-to-one parent links).")


--- Hierarchy QC Summary ---
 docs_total  null_meso_docs  null_macro_docs  micro_clusters  meso_clusters  macro_clusters  micro_without_meso  micro_without_macro  micro_with_multiple_meso  micro_with_multiple_macro  meso_without_macro  meso_with_multiple_macro
     300097               0                0             461             42              12                   0                    0                         0                          0                   0                         0

[ok] Hierarchy mapping checks passed (no nulls and one-to-one parent links).
